# box-array-to-tensor-with-recipe composite — cx9: box raw output with a parents-by-argidx dict built from input args

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `box-array-to-tensor-with-recipe`, `parents-dict-by-argidx`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "box-array-to-tensor-with-recipe"
DD_ATOM_IDS = ["box-array-to-tensor-with-recipe", "parents-dict-by-argidx"]
DD_SUBTOPICS = ["Backprop: Box array as Tensor + recipe", "Backprop: Parents dict by argidx"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Boxing + parents dict in one helper

Boxing wraps the raw output in a MiniTensor and attaches a Recipe. The Recipe's `parents` field is the edge list of the compute graph — and it must be built from the ORIGINAL `args` (with positions preserved) BEFORE we hand control to the boxer.

Composing these two atoms: walk `args` once with `isinstance(a, MiniTensor)` to build `parents`, then box `out_raw` with `Recipe(fwd_fn, raw_args, kwargs, parents)`. The argidx in the dict KEYS is the most failure-prone detail — renumbering breaks the reverse pass's BACK_FUNCS lookup.

### Composite Exercise — box raw output with a parents-by-argidx dict built from input args

**Atoms exercised together**: `box-array-to-tensor-with-recipe`, `parents-dict-by-argidx`

Implement `cx9_box_with_parents(out_raw, fwd_fn, args, raw_args, kwargs)`. Build the parents dict from `args` (the ORIGINAL, pre-unbox tuple — mixed MiniTensor + scalars), then box `out_raw` into a `MiniTensor` with `requires_grad=True` and a 4-field Recipe attached.

Rules for the parents dict:
- Use `isinstance(a, MiniTensor)` to filter — scalars, raw `torch.Tensor`, tuples are all skipped.
- KEEP the original `argidx` as the dict key. Do NOT collapse `(0.5, x, y)` to `{0: x, 1: y}` — it must be `{1: x, 2: y}`.

The Recipe gets `(fwd_fn, raw_args, kwargs, parents)` in that order.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx9_box_with_parents(out_raw, fwd_fn, args, raw_args, kwargs):
    """Box raw output with parents-by-argidx and a Recipe attached."""
    raise NotImplementedError

def _test_cx9():
    # --- single MiniTensor at arg-0 ---
    x = MiniTensor(t.tensor([1.0, 2.0, 3.0]))
    raw_out = x.array * 2
    out = cx9_box_with_parents(raw_out, t.multiply, (x, 2.0), (x.array, 2.0), {})
    assert isinstance(out, MiniTensor)
    assert out.array is raw_out
    assert out.requires_grad is True
    assert out.recipe is not None
    assert out.recipe.func is t.multiply
    assert out.recipe.args == (x.array, 2.0)
    assert out.recipe.kwargs == {}
    assert out.recipe.parents == {0: x}, f'scalar at arg-1 must be skipped: {out.recipe.parents}'
    # --- non-adjacent MiniTensors: float, T, T ---
    a = MiniTensor(t.tensor([1.0]))
    b = MiniTensor(t.tensor([2.0]))
    def _f(s, x, y): return s * (x + y)
    raw_out = _f(0.5, a.array, b.array)
    out = cx9_box_with_parents(raw_out, _f, (0.5, a, b), (0.5, a.array, b.array), {})
    assert out.recipe.parents == {1: a, 2: b}, 'argidx must be PRESERVED, not collapsed to {0:a, 1:b}'
    assert out.recipe.parents[1] is a
    assert out.recipe.parents[2] is b
    # --- all-scalar args ---
    raw_out = t.tensor(5.0)
    out = cx9_box_with_parents(raw_out, t.add, (2.0, 3.0), (2.0, 3.0), {})
    assert out.recipe.parents == {}, 'no MiniTensors → empty parents'
    assert out.recipe is not None, 'empty parents still produces a Recipe'
    # --- raw torch.Tensor must be SKIPPED from parents ---
    raw_pass = t.tensor([9.0])
    x = MiniTensor(t.tensor([3.0]))
    raw_out = raw_pass * x.array
    out = cx9_box_with_parents(raw_out, t.multiply, (raw_pass, x), (raw_pass, x.array), {})
    assert out.recipe.parents == {1: x}, 'raw torch.Tensor is not a MiniTensor → skipped'
    # --- five-position arg sweep ---
    t1 = MiniTensor(t.tensor([1.0]))
    t2 = MiniTensor(t.tensor([2.0]))
    t3 = MiniTensor(t.tensor([3.0]))
    args5 = (t1, 'x', t2, 7, t3)
    raw5 = t.tensor([1.0])
    out = cx9_box_with_parents(raw5, t.add, args5, args5, {})
    assert out.recipe.parents == {0: t1, 2: t2, 4: t3}, f'argidx-preserved across gaps: {out.recipe.parents}'
    _dd_passed.add('cx9')

_test_cx9()

<details><summary>Show solution — cx9</summary>

```python
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx9_box_with_parents(out_raw, fwd_fn, args, raw_args, kwargs):
    parents = {
        idx: a
        for idx, a in enumerate(args)
        if isinstance(a, MiniTensor)
    }
    out = MiniTensor(out_raw, requires_grad=True)
    out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
    return out
```

**`enumerate` BEFORE `if`.** The `for idx, a in enumerate(args) if isinstance(a, MiniTensor)` pattern attaches the position first, then filters — which is why argidx is preserved across gaps. Filtering first then re-enumerating would collapse positions and break the downstream BACK_FUNCS lookup.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx9',
        'subtopics': ["Backprop: Box array as Tensor + recipe", "Backprop: Parents dict by argidx"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()